In [ ]:
!nvidia-smi

Tue May 20 16:29:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install opencv-python

In [ ]:
!apt-get install libgl1-mesa-glx -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import itertools
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
!pip install -U -q kaggle

from google.colab import files
files.upload()

Saving kaggle (3).json to kaggle (3) (2).json


{'kaggle (3) (2).json': b'{"username":"manojzyn","key":"24d2cf92adbb6405368d15cc5bf4aa6b"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other


In [ ]:
!unzip chest-xray-pneumonia.zip

Archive:  chest-xray-pneumonia.zip
replace chest_xray/__MACOSX/._chest_xray? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls chest_xray/train

In [ ]:
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory("chest_xray/train", label_mode="categorical",
                                                               class_names=CLASS_NAMES, image_size=(224, 224),
                                                               shuffle=True, seed=42, batch_size=None)
test_ds = tf.keras.preprocessing.image_dataset_from_directory("chest_xray/test", label_mode="categorical",
                                                              class_names=CLASS_NAMES, image_size=(224, 224),
                                                              shuffle=True, seed=42, batch_size=None)
val_ds = tf.keras.preprocessing.image_dataset_from_directory("chest_xray/val", label_mode="categorical",
                                                             class_names=CLASS_NAMES, image_size=(224, 224),
                                                             shuffle=True, seed=42, batch_size=None)

In [ ]:
row, col = 4, 4
fig, axes = plt.subplots(row, col, figsize=(20, 20))

for index, x in enumerate(train_ds):
    axes[index // row, index % col].imshow(x[0].numpy().astype(int))
    axes[index // row, index % col].set_title(CLASS_NAMES[x[1].numpy().argmax()])

    if index == (row * col) - 1:
        break

plt.show()

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = [256, 256]

def func(image, label):
    return tf.image.resize(image, IMG_SIZE), label

train = train_ds.map(func).batch(BATCH_SIZE).shuffle(1000).prefetch(tf.data.AUTOTUNE).cache()
test = test_ds.map(func).batch(BATCH_SIZE).shuffle(1000).prefetch(tf.data.AUTOTUNE)
val = val_ds.map(func).batch(BATCH_SIZE).shuffle(1000).prefetch(tf.data.AUTOTUNE)

In [ ]:
def create_model():
    efficient_net = tf.keras.applications.VGG19(include_top=False, weights="imagenet", input_shape=IMG_SIZE + [3])
    efficient_net.trainable = False
    x = tf.keras.layers.GlobalAveragePooling2D()(efficient_net.output)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    model = tf.keras.Model(efficient_net.input, outputs)

    model.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"]
    )
    return model

In [ ]:
model = create_model()

In [ ]:
model.summary()

In [ ]:
class ModelCheckpointCustom(tf.keras.callbacks.Callback):

    def __init__(self, model_path, save_best_only=False):
        super(ModelCheckpointCustom, self).__init__()

        self.model_path = model_path
        self.save_best_only = save_best_only
        self.best_val_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):

        current_val_loss = 0 if logs['val_loss'] is None else logs['val_loss']

        if self.save_best_only and current_val_loss < self.best_val_loss:
            self.best_val_loss = current_val_loss
            self.model.save_weights(self.model_path)
        else:
            self.model.save_weights(self.model_path)

save_callbacks = ModelCheckpointCustom(model_path='saved_weights/model.weights.h5', save_best_only=True)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="loss", patience=5)

In [ ]:
import os

# Make sure the directory exists
os.makedirs("saved_weights", exist_ok=True)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Calculate class weights
labels = []
for _, label in train_ds:
    labels.append(np.argmax(label.numpy()))
class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(class_weights))


In [ ]:
EPOCHS = 10
history = model.fit(
    train,
    epochs=EPOCHS,
    validation_data=val,
    validation_steps=int(len(val) * 0.2),
    callbacks=[early_stopping, save_callbacks],
    class_weight=class_weights
)

In [ ]:
pd.DataFrame(history.history).plot()

# Test the model

In [ ]:
loaded_model = create_model()

In [ ]:
loaded_model.load_weights("saved_weights/model.weights.h5")

In [ ]:
y_pred = loaded_model.evaluate(test)

In [ ]:
print("Testing accuracy:", y_pred[1])

In [ ]:
model.save('vgg19_model.h5')

# XAI

In [ ]:
class GradCAM(object):

    def __init__(self, model, alpha=0.8, beta=0.3):

        self.model = model
        self.alpha = alpha
        self.beta = beta

    def apply_heatmap(self, heatmap, image):
        # Resize the heatmap to match the size of the original image
        heatmap = cv2.resize(heatmap, image.shape[:-1])

        # Apply color map (JET) to the heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)

        # Combine the original image and heatmap
        superimposed_img = cv2.addWeighted(np.array(image).astype(np.float32), self.alpha,
                                           np.array(heatmap).astype(np.float32), self.beta, 0)

        return np.array(superimposed_img).astype(np.uint8)

    def gradCAM(self, x_test=None, name='black_max_pool_2', index_class=0):

        with tf.GradientTape() as tape:
             # Get the specified layer from the model
            last_conv_layer = self.model.get_layer(name)

            # Create a new model that outputs the predicted class and the output of the specified layer
            grad_model = tf.keras.Model([self.model.input], [self.model.output, last_conv_layer.output])

            # Get the model predictions and the output of the specified layer for the input image
            model_out, last_conv_layer = grad_model(np.expand_dims(x_test, axis=0))

            # Extract the predicted class output
            class_out = model_out[:, index_class]

            # Compute the gradients of the predicted class output with respect to the last convolutional layer
            grads = tape.gradient(class_out, last_conv_layer)

            # Compute the average gradient values across the spatial dimensions (8x8) and the channels (1152)
            pooled_grads = tf.reduce_mean(grads, axis=(0, 3)) # (1, 8, 8, 1152) -> (8, 8)

            # Compute the average activation values across the spatial dimensions (8x8) and the channels (1152)
            last_conv_layer = tf.reduce_mean(last_conv_layer, axis=(0, 3))

        # Element-wise multiplication of the gradients and the activation values
        heatmap = tf.multiply(pooled_grads, last_conv_layer)

        # Set negative values to zero
        heatmap = np.maximum(heatmap, 0)

        # Normalize the heatmap values between 0 and 1
        heatmap /= np.max(heatmap)
        heatmap = np.array(heatmap)

        # Apply the heatmap on the input image and return the result
        return self.apply_heatmap(heatmap, x_test)

In [ ]:
gradCam = GradCAM(loaded_model, alpha=0.8, beta=0.2)

In [ ]:
# Number of rows and columns in the grid of images
row, col = 4, 4

# Create a figure and subplots grid
fig, axes = plt.subplots(row, col, figsize=(20, 20))

# Iterate over a subset of the test dataset
for index, x in enumerate(test_ds.take(row * col)):

    # Unpack the input and label from the current iteration
    x_test, y_test = x

    # Resize the input image to the specified size
    x_test = tf.image.resize(x_test, IMG_SIZE)

    # Perform prediction on the resized image
    y_pred = loaded_model.predict(tf.expand_dims(x_test, axis=0), verbose=0).argmax()

    # Generate GradCAM heatmap
    grad_heatmap = gradCam.gradCAM(x_test, name='block5_pool', index_class=y_test.numpy().argmax())

    # Display the GradCAM heatmap in the corresponding subplot
    axes[index // row, index % col].imshow(grad_heatmap)

    # Set the title for the subplot
    axes[index // row, index % col].set_title(f"{CLASS_NAMES[y_pred]}/{CLASS_NAMES[y_test.numpy().argmax()]}")

    # Break the loop if all subplots have been filled
    if index == (row * col) - 1:
        break

# Display the figure with subplots
plt.show()

In [ ]:
def conf_matrix(y_test=None, y_pred=None, class_names=None):

    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(8, 8))
    a = ax.matshow(cm, cmap=plt.cm.Greens)

    fig.colorbar(a)

    ax.set(title="Confusion Matrix",
             xlabel="Predicted label",
             ylabel="Actual label",
             xticks=np.arange(len(class_names)),
             yticks=np.arange(len(class_names)),
             xticklabels=class_names,
             yticklabels=class_names)

    ax.xaxis.set_label_position("bottom")
    ax.xaxis.tick_bottom()

    plt.xticks(rotation=60, fontsize=20)
    plt.yticks(fontsize=20)

    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, f"{cm[i, j]}",
                  horizontalalignment="center", color="black" if cm[i, j] < (cm.max() + cm.min()) / 2 else "white",
                  fontsize=12)

In [ ]:
y_pred = []
y_truth = []

for x, y in test:
    y_pred.extend(loaded_model.predict([x], verbose=0).round())
    y_truth.extend(y.numpy())

y_true = np.array(y_truth).argmax(axis=1)
y_prediction = np.array(y_pred).argmax(axis=1)

In [ ]:
# Compute evaluation metrics
accuracy = accuracy_score(y_true, y_prediction)
precision = precision_score(y_true, y_prediction, average='macro')
recall = recall_score(y_true, y_prediction, average='macro')
f1 = f1_score(y_true, y_prediction, average='macro')
confusion_mat = confusion_matrix(y_true, y_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("\nConfusion Matrix:")
print(pd.DataFrame(confusion_mat))

In [ ]:
conf_matrix(y_test=y_true, y_pred=y_prediction, class_names=CLASS_NAMES)

# Lime

In [ ]:
pip install lime


In [ ]:
def predict_fn(images):
    images = tf.image.resize(images, IMG_SIZE)
    preds = loaded_model.predict(images)
    return preds


In [ ]:
import random
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Get 1 sample image and label from the test dataset
for img, label in test_ds.take(5):
    sample_img = img.numpy()
    true_label = label.numpy().argmax()

# Resize to match model input size
sample_img = tf.image.resize(sample_img, IMG_SIZE).numpy().astype(np.uint8)


In [ ]:
explainer = lime_image.LimeImageExplainer()

# Explain prediction on the image
explanation = explainer.explain_instance(
    image=sample_img,
    classifier_fn=predict_fn,
    top_labels=2,
    hide_color=0,
    num_samples=1000  # Number of perturbed samples
)

# Get image and mask for the top predicted class
lime_img, mask = explanation.get_image_and_mask(
    label=explanation.top_labels[0],
    positive_only=True,
    hide_rest=False,
    num_features=10,
    min_weight=0.0
)

# Show explanation
plt.figure(figsize=(6,6))
plt.imshow(mark_boundaries(lime_img, mask))
plt.title(f"LIME Explanation for {CLASS_NAMES[true_label]}")
plt.axis('off')
plt.show()


In [ ]:
def run_lime_with_config(sample_img, predict_fn, label_index=0,
                         positive_only=True, hide_rest=False, num_features=10, num_samples=1000,
                         title="LIME Output"):
    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        image=sample_img,
        classifier_fn=predict_fn,
        top_labels=1,
        hide_color=0,
        num_samples=num_samples
    )

    lime_img, mask = explanation.get_image_and_mask(
        label=explanation.top_labels[label_index],
        positive_only=positive_only,
        hide_rest=hide_rest,
        num_features=num_features
    )

    plt.figure(figsize=(6, 6))
    plt.imshow(mark_boundaries(lime_img, mask))
    plt.title(f"{title}\nFeatures: {num_features}, PosOnly: {positive_only}, HideRest: {hide_rest}, Samples: {num_samples}")
    plt.axis('off')
    plt.show()


In [ ]:
run_lime_with_config(sample_img, predict_fn,
                     positive_only=True, hide_rest=False, num_features=10, num_samples=1000,
                     title="Standard LIME View")


In [ ]:
run_lime_with_config(sample_img, predict_fn,
                     positive_only=True, hide_rest=True, num_features=10, num_samples=1000,
                     title="Isolated Positive Influence")


In [ ]:
run_lime_with_config(sample_img, predict_fn,
                     positive_only=False, hide_rest=False, num_features=10, num_samples=1000,
                     title="All Influences (Pos+Neg)")


In [ ]:
run_lime_with_config(sample_img, predict_fn,
                     positive_only=True, hide_rest=False, num_features=25, num_samples=1000,
                     title="Detailed View (Top 25)")


In [ ]:
import matplotlib.pyplot as plt
from lime import lime_image
from skimage.segmentation import mark_boundaries

def analyze_with_lime(sample_img, predict_fn, class_label=0):
    explainer = lime_image.LimeImageExplainer()

    # Generate explanation
    explanation = explainer.explain_instance(
        image=sample_img,
        classifier_fn=predict_fn,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )

    # Prepare 5 variants with different parameters
    configs = [
        {"positive_only": True, "hide_rest": False, "num_features": 10, "title": "Positive-Only (Top 10), Background Shown"},
        {"positive_only": True, "hide_rest": True,  "num_features": 10, "title": "Positive-Only (Top 10), Background Hidden"},
        {"positive_only": False, "hide_rest": False, "num_features": 10, "title": "All Influences (Top 10)"},
        {"positive_only": True, "hide_rest": False, "num_features": 25, "title": "High Resolution (Top 25 Features)"},
    ]

    # Add low-sample explanation separately
    explanation_low_sample = explainer.explain_instance(
        image=sample_img,
        classifier_fn=predict_fn,
        top_labels=2,
        hide_color=0,
        num_samples=100  # Low sample test
    )
    lime_img_low, mask_low = explanation_low_sample.get_image_and_mask(
        label=explanation_low_sample.top_labels[0],
        positive_only=True,
        hide_rest=False,
        num_features=10
    )

    # Plot grid
    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    axs = axs.flatten()

    for idx, cfg in enumerate(configs):
        lime_img, mask = explanation.get_image_and_mask(
            label=explanation.top_labels[class_label],
            positive_only=cfg["positive_only"],
            hide_rest=cfg["hide_rest"],
            num_features=cfg["num_features"]
        )
        axs[idx].imshow(mark_boundaries(lime_img, mask))
        axs[idx].set_title(cfg["title"])
        axs[idx].axis('off')

    axs[4].imshow(mark_boundaries(lime_img_low, mask_low))
    axs[4].set_title("Low Samples (100 perturbed)")
    axs[4].axis('off')

    axs[5].axis('off')  # Empty subplot

    plt.suptitle("LIME Analysis - Various Perspectives", fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
analyze_with_lime(sample_img, predict_fn)
